In [1]:
# 1. 데이터 생성

In [ ]:
import os, json, time, re, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document


load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
llm.invoke('안녕하세요')

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 9, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVYF2RNeV4cZDqCIxxbwcKlqsBV1h', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9a6a-7dd9-7d71-b423-aabb9aebcd01-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 10, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
import FinanceDataReader as fdr
etf_listing =fdr.StockListing('ETF/KR')

In [5]:
!pwd

/data/lecture/modu/llm/notebooks/week06


In [6]:
etf_listing.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,94050,5,-660,-0.70,94029.0,33.9969,10221423,962685,213541
1,360750,4,TIGER 미국S&P500,25940,2,160,0.62,25977.0,1.3441,11532078,298601,161023
2,396500,2,TIGER 반도체TOP10,35735,5,-200,-0.56,35759.0,49.1078,8942100,319898,95198
3,133690,4,TIGER 미국나스닥100,172915,2,920,0.53,173385.0,2.7072,424825,73348,87460
4,379800,4,KODEX 미국S&P500,23705,2,145,0.62,23743.0,1.2831,20001113,473308,84852


In [7]:
etf_listing['Symbol'].astype(str).str.zfill(6)

0       069500
1       360750
2       396500
3       133690
4       379800
         ...  
1088    412560
1089    334700
1090    465620
1091    306520
1092    301410
Name: Symbol, Length: 1093, dtype: object

In [8]:
name_by_ticker = dict(zip(etf_listing['Symbol'].astype(str).str.zfill(6), etf_listing['Name']))
name_by_ticker

{'069500': 'KODEX 200',
 '360750': 'TIGER 미국S&P500',
 '396500': 'TIGER 반도체TOP10',
 '133690': 'TIGER 미국나스닥100',
 '379800': 'KODEX 미국S&P500',
 '102110': 'TIGER 200',
 '459580': 'KODEX CD금리액티브(합성)',
 '488770': 'KODEX 머니마켓액티브',
 '229200': 'KODEX 코스닥150',
 '379810': 'KODEX 미국나스닥100',
 '278530': 'KODEX 200TR',
 '122630': 'KODEX 레버리지',
 '411060': 'ACE KRX금현물',
 '310970': 'TIGER MSCI Korea TR',
 '357870': 'TIGER CD금리투자KIS(합성)',
 '091160': 'KODEX 반도체',
 '498400': 'KODEX 200타겟위클리커버드콜',
 '233740': 'KODEX 코스닥150레버리지',
 '423160': 'KODEX KOFR금리액티브(합성)',
 '381180': 'TIGER 미국필라델피아반도체나스닥',
 '381170': 'TIGER 미국테크TOP10 INDXX',
 '273130': 'KODEX 종합채권(AA-이상)액티브',
 '148020': 'RISE 200',
 '360200': 'ACE 미국S&P500',
 '458730': 'TIGER 미국배당다우존스',
 '0043B0': 'TIGER 머니마켓액티브',
 '367380': 'ACE 미국나스닥100',
 '102780': 'KODEX 삼성그룹',
 '481050': 'KODEX CD1년금리플러스액티브(합성)',
 '161510': 'PLUS 고배당주',
 '455890': 'RISE 머니마켓액티브',
 '449170': 'TIGER KOFR금리액티브(합성)',
 '292150': 'TIGER 코리아TOP10',
 '395160': 'KODEX AI반도체',
 '487240': 'K

In [9]:
tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
    "132030": {"category": "원자재",   "expense_ratio": 0.68, "dividend_yield": 0.0,
               "keywords": ["금", "원자재", "인플레이션", "헤지", "안전자산"]},
    "167860": {"category": "채권",     "expense_ratio": 0.15, "dividend_yield": 2.8,
               "keywords": ["채권", "국고채", "안전", "이자", "저위험"]},
    "214980": {"category": "채권",     "expense_ratio": 0.05, "dividend_yield": 3.2,
               "keywords": ["단기채", "안전", "예금대안", "저위험", "이자"]},
    "329200": {"category": "부동산",   "expense_ratio": 0.29, "dividend_yield": 4.2,
               "keywords": ["리츠", "부동산", "배당", "인프라", "실물자산"]},
    "284430": {"category": "혼합",     "expense_ratio": 0.09, "dividend_yield": 1.5,
               "keywords": ["혼합", "자산배분", "균형", "중위험", "분산투자"]},
    "371460": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["중국", "전기차", "테마", "해외주식", "고위험"]},
}

In [10]:
for t, info in tickers_info.items():
    name = name_by_ticker.get(t, "(이름 미확인)")
    print(f" {t} | {name:30s} | {info['category']}")

 069500 | KODEX 200                      | 국내주식
 379800 | KODEX 미국S&P500                 | 해외주식
 411060 | ACE KRX금현물                     | 배당
 305540 | TIGER 2차전지테마                   | 테마
 379810 | KODEX 미국나스닥100                 | 해외주식
 381180 | TIGER 미국필라델피아반도체나스닥            | 해외주식
 132030 | KODEX 골드선물(H)                  | 원자재
 167860 | KIWOOM 국고채10년레버리지              | 채권
 214980 | KODEX 단기채권PLUS                 | 채권
 329200 | TIGER 리츠부동산인프라                 | 부동산
 284430 | KODEX 200미국채혼합                 | 혼합
 371460 | TIGER 차이나전기차SOLACTIVE          | 해외주식


In [11]:
def collect_etf_price(ticker, days= 365):
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = ( datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    df = fdr.DataReader(ticker, start_date, end_date)
    return {'status' : 'real', 'data' : df, 'ticker' : ticker}

In [12]:
def compute_metrics(df):
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 -1)*100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return {'return_1y' : round(ann_ret, 2), 'volatility' : round(ann_vol,2), 'mdd' : round(mdd, 2)}

In [13]:
def risk_from_vol(vol):
    if vol < 5 : return "낮음"
    if vol < 15: return "약간 낮음"
    if vol < 25: return "중간"
    return "높음"

In [14]:
tickers_info

{'069500': {'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식']},
 '379800': {'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식']},
 '411060': {'category': '배당',
  'expense_ratio': 0.01,
  'dividend_yield': 3.5,
  'keywords': ['미국', '배당', '배당성장', '저비용', '안정']},
 '305540': {'category': '테마',
  'expense_ratio': 0.45,
  'dividend_yield': 0.0,
  'keywords': ['2차전지', '배터리', '테마', '성장', '고위험']},
 '379810': {'category': '해외주식',
  'expense_ratio': 0.07,
  'dividend_yield': 0.0,
  'keywords': ['미국', '나스닥', '기술주', '성장', 'IT']},
 '381180': {'category': '해외주식',
  'expense_ratio': 0.49,
  'dividend_yield': 0.0,
  'keywords': ['반도체', 'AI', '미국', '기술주', '고위험']},
 '132030': {'category': '원자재',
  'expense_ratio': 0.68,
  'dividend_yield': 0.0,
  'keywords': ['금', '원자재', '인플레이션', '헤지', '안전자산']},
 '167860': {'category': '채권',
  'expense_ratio': 0.15,
  'dividend_yiel

In [15]:
etf_data =[]
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    # {'status' : 'real', 'data' : df, 'ticker' : ticker}
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker" : t,
        "name" : name,
        **info,
        **metrics,
        'risk_level' : risk_from_vol(metrics['volatility']),
        'data_staus' : result['status']
    })

In [16]:
etf_data[:2]

[{'ticker': '069500',
  'name': 'KODEX 200',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
  'return_1y': np.float64(221.8),
  'volatility': np.float64(36.6),
  'mdd': np.float64(-20.62),
  'risk_level': '높음',
  'data_staus': 'real'},
 {'ticker': '379800',
  'name': 'KODEX 미국S&P500',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식'],
  'return_1y': np.float64(42.41),
  'volatility': np.float64(13.32),
  'mdd': np.float64(-5.7),
  'risk_level': '약간 낮음',
  'data_staus': 'real'}]

In [17]:
def generate_description(etf):
    
    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}$
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%
    """
    
    return llm.invoke([{'role' : 'user', 'content' : prompt}]).content.strip()

In [18]:
documents = []
for etf in etf_data:
    desc = generate_description(etf)
    text = (f"{etf['name']} ({etf['category']}): {desc} "
            f"키워드: {', '.join(etf['keywords'])}"
            f"수수료 {etf['expense_ratio']}%, 배당수익률: {etf['dividend_yield']}%"
            f"수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%")
    
    metadata = {k:v for k, v in etf.items() if k!="keywords"}
    metadata['keywords'] = ', '.join(etf['keywords'])
    
    documents.append(Document(page_content=text, metadata = metadata))
    

In [19]:
documents

[Document(metadata={'ticker': '069500', 'name': 'KODEX 200', 'category': '국내주식', 'expense_ratio': 0.15, 'dividend_yield': 1.8, 'return_1y': np.float64(221.8), 'volatility': np.float64(36.6), 'mdd': np.float64(-20.62), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '코스피, 대형주, 인덱스, 분산투자, 국내주식'}, page_content='KODEX 200 (국내주식): KODEX 200은 코스피 대형주에 투자하는 국내주식 인덱스 ETF로, 분산투자를 통해 안정성을 추구하며, 1년 수익률이 221.8%로 높은 성과를 기록하고 있습니다. 수수료는 0.15%이며, 배당수익률은 1.8%입니다. 키워드: 코스피, 대형주, 인덱스, 분산투자, 국내주식수수료 0.15%, 배당수익률: 1.8%수익률 : 221.8% / 변동성: 36.6% / MDDD : -20.62%'),
 Document(metadata={'ticker': '379800', 'name': 'KODEX 미국S&P500', 'category': '해외주식', 'expense_ratio': 0.05, 'dividend_yield': 0.0, 'return_1y': np.float64(42.41), 'volatility': np.float64(13.32), 'mdd': np.float64(-5.7), 'risk_level': '약간 낮음', 'data_staus': 'real', 'keywords': '미국, S&P500, 대형주, 성장, 해외주식'}, page_content='KODEX 미국S&P500 (해외주식): KODEX 미국S&P500 ETF는 미국의 S&P500 지수를 추종하는 해외주식 ETF로, 대형주 중심의 성장주에 투자합니다. 수수료는 0.05%이며, 최근 1년간 수익률은 4

In [20]:
vectorstore = LangchainFAISS.from_documents(documents, embeddings)

In [21]:
vectorstore.index.ntotal

12

In [22]:
query = '미국 기술주에 투자하고 싶어요'
vectorstore.similarity_search_with_score(query, k=3)

[(Document(id='73ca1e4d-59d1-41c5-ae91-63f4e7b30a9d', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(169.62), 'volatility': np.float64(31.34), 'mdd': np.float64(-10.35), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 AI 관련 기술주에 투자하는 고위험 해외주식 ETF로, 최근 1년 동안 169.62%의 높은 수익률을 기록했습니다. 수수료는 0.49%이며, 배당수익률은 0%로 변동성이 31.34%로 나타나고 있습니다. 키워드: 반도체, AI, 미국, 기술주, 고위험수수료 0.49%, 배당수익률: 0.0%수익률 : 169.62% / 변동성: 31.34% / MDDD : -10.35%'),
  np.float32(1.1911612)),
 (Document(id='53a67661-31a2-4b5a-b6aa-d3fec5318afa', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(53.33), 'volatility': np.float64(16.58), 'mdd': np.float64(-7.34), 'risk_level': '중간', 'data_staus': 'real', 'keywords': '미국, 나스닥, 기술

In [23]:
etf_knowledge_base = [
    {"ticker": "069500", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
     "risk": "중간", "expense_ratio": 0.15},
    {"ticker": "379800", "name": "KODEX 미국S&P500TR", "category": "해외주식",
     "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
     "risk": "중간", "expense_ratio": 0.05},
    {"ticker": "461460", "name": "KODEX 미국나스닥100TR", "category": "해외주식",
     "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
     "risk": "높음", "expense_ratio": 0.05},
]

In [24]:
etf_names = [e['name'] + ': ' + e['description'] for e in etf_knowledge_base]
etf_names

['KODEX 200: KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.',
 'KODEX 미국S&P500TR: S&P500 추적, 배당 자동 재투자. 수수료 0.05%.',
 'KODEX 미국나스닥100TR: 나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.']

In [25]:
prompt = f"""다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요

ETF 목록:
{json.dumps(etf_names, ensure_ascii=False, indent=2)}

형식 : 번호. 질문"""

synthetic = llm.invoke(prompt)

In [26]:
synthetic

AIMessage(content='1. KODEX 200 ETF는 어떤 종류의 주식에 투자하나요? 대형주 중심이라고 했는데, 구체적으로 어떤 기업들이 포함되어 있나요?\n\n2. KODEX 미국S&P500TR ETF의 배당 자동 재투자 기능은 어떻게 작동하나요? 배당금이 어떻게 처리되는지 궁금합니다.\n\n3. KODEX 미국나스닥100TR ETF는 높은 성장성과 변동성을 가진다고 했는데, 이 ETF에 투자할 때 어떤 리스크를 고려해야 하나요?\n\n4. KODEX 200 ETF와 KODEX 미국S&P500TR ETF의 수수료 차이가 투자 성과에 어떤 영향을 미칠까요?\n\n5. KODEX 미국나스닥100TR ETF에 투자할 경우, 기술주 중심의 포트폴리오가 어떤 장점과 단점을 가지고 있는지 설명해 주실 수 있나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 193, 'prompt_tokens': 144, 'total_tokens': 337, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DVYFZHjsUAmOerlPv1IKxdKNO1EoG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9a6

In [27]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

In [28]:
kiwi = Kiwi()

In [29]:
def kiwi_tokenize(text):
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        if token.tag in ("SL", "SN", "NNG", "NNP"):
            result.append(token.form.lower())
            
    return result

In [30]:
kiwi_tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?")

['kodex', '200', 'etf', '종류', '주식', '투자']

In [31]:
test_text = "배당수익률 3% 이상인 미국 ETF를 추천해주세요"
kiwi_tokenize(test_text)

['배당', '수익', '3', '이상', '미국', 'etf', '추천']

In [32]:
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]

In [33]:
bm25 = BM25Okapi(corpus)

In [34]:
def bm25_search(query, k=5):
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]
    

In [35]:
test_queries = [
    "KODEX 200",
    "안전한 투자 상품",
    "배당 ETF",
    "인플레이션 방어"
]

In [36]:
for q in test_queries:
    v_result = vectorstore.similarity_search(q, k=1)
    b_result = bm25_search(q, k=1)
    v_name = v_result[0].metadata['name'] if v_result else "-"
    b_name = b_result[0][0].metadata['name'] if b_result else "-"
    print(f"{q:20s} | {v_name:25s} | {b_name:25s}")

KODEX 200            | KODEX 200                 | KODEX 200                
안전한 투자 상품            | KODEX 단기채권PLUS            | KODEX 단기채권PLUS           
배당 ETF               | ACE KRX금현물                | ACE KRX금현물               
인플레이션 방어             | KIWOOM 국고채10년레버리지         | KODEX 골드선물(H)            


In [37]:
# bm25 : TF-IDF

In [38]:
# k1, b 

In [39]:
query = "배당 ETF"
tokens = kiwi_tokenize(query)
params = [(0.5, 0.25), (1.5, 0.75), (2.5, 0.9)]
for k1, b in params:
    bm25_temp = BM25Okapi(corpus, k1=k1, b=b)
    scores = bm25_temp.get_scores(tokens)
    top3 = np.argsort(scores)[::-1][:3]
    print(f"k1 = {k1}, b= {b}")
    for idx in top3:
        name = documents[idx].metadata['name']
        print(f" [{scores[idx]}] {name}")
    

k1 = 0.5, b= 0.25
 [0.897927116360912] TIGER 2차전지테마
 [0.8961684799853846] ACE KRX금현물
 [0.8952681235017367] TIGER 차이나전기차SOLACTIVE
k1 = 1.5, b= 0.75
 [1.1587663948792346] ACE KRX금현물
 [1.078135173591938] TIGER 2차전지테마
 [1.057765717866019] TIGER 차이나전기차SOLACTIVE
k1 = 2.5, b= 0.9
 [1.3634771954524774] ACE KRX금현물
 [1.1806357049724825] TIGER 2차전지테마
 [1.1461085844371377] TIGER 차이나전기차SOLACTIVE


In [40]:
def filtered_search(vectorstore, query, filters=None, k=5, fetch_k=20):  # filters = {"expense_ratio" : 0.1, "category" : "해외주식"}
    results = vectorstore.similarity_search_with_score(query, fetch_k)
    if not filters:
        return results[:k]
    
    filtered = []
    for doc, score in results:
        match = True
        for key, condition in filters.items():
            val = doc.metadata.get(key)  # 0.49
            if isinstance(condition, dict):
                if "less_than" in condition and val > condition["less_than"]:
                    match = False
                if "greater_than" in condition and val < condition["greater_than"]:
                    match = False
            elif val != condition:
                match = False
        if match:
            filtered.append((doc, score))
    
    return filtered[:k]

In [41]:
results = filtered_search(vectorstore, "저비용 해외 ETF", 
                          filters = {"category":"해외주식", "expense_ratio" : {"less_than": 0.5}}, k=3)

In [42]:
results

[(Document(id='73ca1e4d-59d1-41c5-ae91-63f4e7b30a9d', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(169.62), 'volatility': np.float64(31.34), 'mdd': np.float64(-10.35), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 (해외주식): TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 AI 관련 기술주에 투자하는 고위험 해외주식 ETF로, 최근 1년 동안 169.62%의 높은 수익률을 기록했습니다. 수수료는 0.49%이며, 배당수익률은 0%로 변동성이 31.34%로 나타나고 있습니다. 키워드: 반도체, AI, 미국, 기술주, 고위험수수료 0.49%, 배당수익률: 0.0%수익률 : 169.62% / 변동성: 31.34% / MDDD : -10.35%'),
  np.float32(0.9882194)),
 (Document(id='6233d411-4d2a-436d-8d56-e8a45637d74f', metadata={'ticker': '371460', 'name': 'TIGER 차이나전기차SOLACTIVE', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(67.72), 'volatility': np.float64(24.85), 'mdd': np.float64(-11.34), 'risk_level': '중간', 'data_staus': 'real', 'keywords': '중국,

In [50]:
def smart_filtered_search(query):
    prompt = f"""사용자 쿼리에서 ETF 검색 필터를 추출하세요.
    
    쿼리 :  {query}
    
    사용 가능한 필터 필드:
    - category : 국내주식, 해외주식, 배당, 테마 ...
    - risk_level : 낮음, 중간, 높음
    - expense_ratio : 숫자 (% "less_than"/"greater_than")
    - dividend_yield : 숫자 (%)
    - return_1y : 숫자 (%)
    - volatility : 숫자 (%)
    
    JSON 형식으로만 응답하세요. 필터가 없으면 {{}}.
    예. {{"category":"해외주식","expense_ratio":{"less_than":0.5}}}
    """
    
    response = llm.invoke([{'role' : 'user', 'content' : prompt}]).content
    try:
        if "```" in response:
            response = response.split("```")[1].replace("json", "")
        return json.loads(response)
    except Exception:
        return {}

In [51]:
test_query = "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"
filters = smart_filtered_search(test_query)
filters

{'expense_ratio': {'less_than': 0.1}, 'dividend_yield': {'greater_than': 3}}

In [57]:
def smart_document_search(query, k=3):
    filters = smart_filtered_search(query)
    print(f"쿼리 : {query}, {filters}")
    print(f"필터 :{json.dumps(filters, ensure_ascii=False)}")
    
    results = filtered_search(vectorstore, query, filters = filters, k=k)
    if not results:
        print("필터 조건에 맞는 결과가 없습니다. 필터 없이 재검색....")
        results = filtered_search(vectorstore, query, k=k)
    
    print(f"결과  : {len(results)}개")
    for doc, score in results:
        m = doc.metadata
        print(m)
    
    return results

In [58]:
smart_document_search("위험 낮고 배당 2% 이상인 ETF")

쿼리 : 위험 낮고 배당 2% 이상인 ETF, {'risk_level': '낮음', 'dividend_yield': 2}
필터 :{"risk_level": "낮음", "dividend_yield": 2}
필터 조건에 맞는 결과가 없습니다. 필터 없이 재검색....
결과  : 3개
{'ticker': '305540', 'name': 'TIGER 2차전지테마', 'category': '테마', 'expense_ratio': 0.45, 'dividend_yield': 0.0, 'return_1y': np.float64(102.47), 'volatility': np.float64(47.5), 'mdd': np.float64(-23.19), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '2차전지, 배터리, 테마, 성장, 고위험'}
{'ticker': '214980', 'name': 'KODEX 단기채권PLUS', 'category': '채권', 'expense_ratio': 0.05, 'dividend_yield': 3.2, 'return_1y': np.float64(2.42), 'volatility': np.float64(0.21), 'mdd': np.float64(-0.08), 'risk_level': '낮음', 'data_staus': 'real', 'keywords': '단기채, 안전, 예금대안, 저위험, 이자'}
{'ticker': '411060', 'name': 'ACE KRX금현물', 'category': '배당', 'expense_ratio': 0.01, 'dividend_yield': 3.5, 'return_1y': np.float64(59.23), 'volatility': np.float64(32.95), 'mdd': np.float64(-22.76), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '미국, 배당, 배당성장, 저비용, 안정'}


[(Document(id='128f7929-f876-4a55-8784-a6164f462d01', metadata={'ticker': '305540', 'name': 'TIGER 2차전지테마', 'category': '테마', 'expense_ratio': 0.45, 'dividend_yield': 0.0, 'return_1y': np.float64(102.47), 'volatility': np.float64(47.5), 'mdd': np.float64(-23.19), 'risk_level': '높음', 'data_staus': 'real', 'keywords': '2차전지, 배터리, 테마, 성장, 고위험'}, page_content='TIGER 2차전지테마 (테마): TIGER 2차전지테마 ETF는 2차전지 관련 기업에 투자하는 고위험 성장형 테마 ETF로, 최근 1년간 102.47%의 수익률을 기록했습니다. 수수료는 0.45%이며, 배당수익률은 0%로 변동성이 47.5%로 높은 편입니다. 키워드: 2차전지, 배터리, 테마, 성장, 고위험수수료 0.45%, 배당수익률: 0.0%수익률 : 102.47% / 변동성: 47.5% / MDDD : -23.19%'),
  np.float32(0.84988415)),
 (Document(id='9f2d0871-52fb-4c75-af17-ab1b3bcfd0fd', metadata={'ticker': '214980', 'name': 'KODEX 단기채권PLUS', 'category': '채권', 'expense_ratio': 0.05, 'dividend_yield': 3.2, 'return_1y': np.float64(2.42), 'volatility': np.float64(0.21), 'mdd': np.float64(-0.08), 'risk_level': '낮음', 'data_staus': 'real', 'keywords': '단기채, 안전, 예금대안, 저위험, 이자'}, page_content='KODEX 단기채권PLUS

In [61]:
# 하이브리드 검색 : 벡터검색  + 키워드검색
def hybrid_search(query, alpha=0.5, k=5, filters=None):
    
    vec_results = vectorstore.similarity_search_with_score(query, k=20)   
    vec_scores = {} # 0 ~ 1   0.5  -> 0.5   
    for doc, dist in vec_results:
        vec_scores[doc.metadata['name']] = 1 / (1+dist)
    # vec_scores = {'TIGER 2차전지테마' : 0.5, 'TIGER 3차전지테마' : 0.9}
    
    bm25_results = bm25_search(query, k=20)
    bm25_scores = {} # 1 ~ 2   1.5  -> 0.5
    for doc, score in bm25_results:
        bm25_scores[doc.metadata['name']] = score
        
    def normalize(scores):
        if not scores:
            return scores
        
        vals = list(scores.values())
        min_, max_ = min(vals), max(vals)
        rng = max_ - min_ if max_ != min_ else 1.0
        return {k: (v-min_)/rng    for k, v in scores.items()}
    
    vec_norm = normalize(vec_scores)  # vec_norm = {'TIGER 2차전지테마' : 0.5, 'TIGER 3차전지테마' : 0.9}
    bm25_norm = normalize(bm25_scores) # bm25_norm = {'TIGER 2차전지테마' : 0.3, 'TIGER 3차전지테마' : 0.7}
    
    all_names = set(vec_norm) | set(bm25_norm)
    combined = {}
    for name in all_names:
        v = vec_norm.get(name, 0)
        b = bm25_norm.get(name, 0)
        combined[name] = alpha *  v + (1-alpha) * b  # combined = {'TIGER 2차전지테마' : 0.4, 'TIGER 3차전지테마' : ...}
    
    if filters:
        name_to_doc = {d.metadata['name'] : d  for d in documents}
        filtered = {}
        for name, score in combined.items():
            doc = name_to_doc.get(name)
            if not doc:
                continue
                
            match = True
            for key, condition in filters.items():
                val = doc.metadata.get(key)  
                if isinstance(condition, dict):
                    if "less_than" in condition and val > condition["less_than"]:
                        match = False
                    if "greater_than" in condition and val < condition["greater_than"]:
                        match = False
                elif val != condition:
                    match = False
            if match:
                filtered[name] = score
        combined = filtered
        
    sorted_results = sorted(combined.items(), key=lambda x : x[1], reverse=True)
    return sorted_results[:k]

In [64]:
query = "미국 기술 성장 ETF"
for alpha in [0.0, 0.5, 1.0]:
    results = hybrid_search(query, alpha=alpha, k=3, filters=None)
    print(results)
    names = [r[0] for r in results]
    print(f"alpha : {alpha} : {names}")

[('KODEX 미국나스닥100', np.float64(1.0)), ('TIGER 미국필라델피아반도체나스닥', np.float64(0.7827765491926837)), ('KODEX 미국S&P500', np.float64(0.508247834747415))]
alpha : 0.0 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'KODEX 미국S&P500']
[('KODEX 미국나스닥100', np.float64(1.0)), ('TIGER 미국필라델피아반도체나스닥', np.float64(0.769453652070173)), ('KODEX 미국S&P500', np.float64(0.5678640218105819))]
alpha : 0.5 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'KODEX 미국S&P500']
[('KODEX 미국나스닥100', np.float64(1.0)), ('TIGER 미국필라델피아반도체나스닥', np.float64(0.7561067342758179)), ('KODEX 미국S&P500', np.float64(0.6274545788764954))]
alpha : 1.0 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'KODEX 미국S&P500']


In [63]:
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국필라델피아반도체"]},
    {"query": "2차전지 관련 ETF", "relevant": ["TIGER 2차전지테마"]},
    {"query": "금 투자 인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "국내 대형주 인덱스", "relevant": ["KODEX 200"]},
    {"query": "부동산 배당 투자", "relevant": ["TIGER 리츠부동산인프라"]},
    {"query": "저비용 미국 ETF", "relevant": ["KODEX 미국S&P500TR", "ACE 미국배당다우존스"]},
    {"query": "채권 안정 수익", "relevant": ["TIGER 국고채10년", "KODEX 단기채권PLUS"]},
]

In [65]:
# hybrid_search, vectorstore.similairty~~~, bm25_search
def hit_rate(eval_data, search_fn, k=5):
    hits = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results]
        if any(rel in found for rel in item['relevant']):
            hits += 1
    return hits / len(eval_data)

In [69]:
def mrr(eval_data, search_fn, k=5):
    rr_sum = 0
    for item in eval_data:
        results = search_fn(item['query'], k=k)
        found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results]
        for rank, name in enumerate(found, 1):
            if name in item['relevant']:
                rr_sum += 1/rank
                break
        
#         if any(rel in found for rel in item['relevant']):
#             hits += 1
    return rr_sum / len(eval_data)

In [66]:
def vec_fn(q,k=5):
    results = vectorstore.similarity_search(q, k=k)
    return [(doc.metadata['name'], 1.0) for doc in results]

def bm25_fn(q, k=5):
    results = bm25_search(q, k=k)
    return [(doc.metadata['name'], score) for doc, score in results]

def hybrid_fn(q, k=5):
    return hybrid_search(q, alpha=0.5, k=k)

In [70]:
for name, fn in [("vec", vec_fn), ("bm25", bm25_fn), ("hybrid", hybrid_fn)]:
    hr3 = hit_rate(eval_queries, fn, k=3)
    hr5 = hit_rate(eval_queries, fn, k=5)
    mrr3 = mrr(eval_queries, fn, k=3)
    mrr5 = mrr(eval_queries, fn, k=5)
    print(f"name : {name} | hr3 : {hr3} | hr5 : {hr5} | mrr3 : {mrr3} | mrr5 : {mrr5}")

name : vec | hr3 : 0.625 | hr5 : 0.75 | mrr3 : 0.625 | mrr5 : 0.65625
name : bm25 | hr3 : 0.75 | hr5 : 0.75 | mrr3 : 0.6875 | mrr5 : 0.6875
name : hybrid | hr3 : 0.75 | hr5 : 0.75 | mrr3 : 0.75 | mrr5 : 0.75


In [ ]:
# Reranking

In [ ]:
# MMR (Maximum Marginal Relevance) -> 다양한 문서를 검색

In [72]:
query = "좋은 투자 상품 추천"
sim_results = vectorstore.similarity_search(query, k=5)
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k = 12, lambda_multi=0.5)
# mmr = lambda * similarity + (1-lambda) * diversity

for doc in sim_results:
    print(f" {doc.metadata['name']} | {doc.metadata['category']}")

print("==========================")
for doc in mmr_results:
    print(f" {doc.metadata['name']} | {doc.metadata['category']}")

 TIGER 2차전지테마 | 테마
 KODEX 단기채권PLUS | 채권
 KODEX 200미국채혼합 | 혼합
 ACE KRX금현물 | 배당
 TIGER 리츠부동산인프라 | 부동산
 TIGER 2차전지테마 | 테마
 KODEX 200 | 국내주식
 KIWOOM 국고채10년레버리지 | 채권
 ACE KRX금현물 | 배당
 KODEX 단기채권PLUS | 채권


In [75]:
def llm_rerank(query, candidates, k=5):
    name_to_doc = {d.metadata['name'] : d for d in documents}
    cand_text = ""
    for i, (name, score) in enumerate(candidates):
        doc = name_to_doc.get(name)
        if doc:
            m = doc.metadata
            cand_text += (f"{i+1}. {name} ({m['category']}) |"
                         f"수수료 {m['expense_ratio']}% | 수익률 {m['return_1y']}% | "
                          f"배당 {m['dividend_yield']}%\n")
    prompt = f"""사용자 쿼리에 가장 적합한 ETF를 순위대로 정렬하세요.
    
    쿼리 : {query}
    후보 ETF : {cand_text}
    가장 적합한 {k}개를 순위대로 번호만 응답하세요.
    예 : 3, 1, 5, 2, 4"""
    
    response = llm.invoke([{"role" : "user", "content" : prompt}]).content
    numbers = [int(x) for x in re.findall(r'\d+', response)]
    
    reranked = []
    seen = set()
    for n in numbers:
        if 1 <= n <= len(candidates) and n not in seen:
            reranked.append(candidates[n-1])
            seen.add(n)
            
    for c in candidates:
        if c not in reranked:
            reranked.append(c)
    
    return reranked[:k]

In [76]:
query = "초보자에게 안전한 ETF"
initial = hybrid_search(query, alpha=0.5, k=8)
reranked = llm_rerank(query, initial, k=5)

In [77]:
initial

[('KODEX 단기채권PLUS', np.float64(0.9868648251447245)),
 ('KIWOOM 국고채10년레버리지', np.float64(0.6932708911172552)),
 ('KODEX 골드선물(H)', np.float64(0.6080888137221336)),
 ('KODEX 200미국채혼합', np.float64(0.3896769324863797)),
 ('TIGER 2차전지테마', np.float64(0.2115126745474549)),
 ('TIGER 미국필라델피아반도체나스닥', np.float64(0.1760291341142296)),
 ('TIGER 차이나전기차SOLACTIVE', np.float64(0.17381303353197103)),
 ('KODEX 200', np.float64(0.08330609219138056))]

In [78]:
reranked

[('KODEX 단기채권PLUS', np.float64(0.9868648251447245)),
 ('KIWOOM 국고채10년레버리지', np.float64(0.6932708911172552)),
 ('KODEX 200미국채혼합', np.float64(0.3896769324863797)),
 ('KODEX 골드선물(H)', np.float64(0.6080888137221336)),
 ('TIGER 2차전지테마', np.float64(0.2115126745474549))]

In [ ]:
# contents-based filtering
# A : 짜장면 : => 짜장면 맛집
# B : 탕수육 : => 탕수육 맛집
# C : 쟁반짜장 : => 짜장면 맛집

# A : 안전추구형 => 안전한 상품만 추천
# B : 높은 수익율 => 

In [ ]:
#       Collaborative Filtering (협업 기반 추천 , 협업 기반 필터링)
#          1    2    3     4    5   6     7
# A        O    O    O
# B                       O     O
# C        O    O    ?

In [79]:
etf_data = [
    {"name": "KODEX 200",             "category": "국내주식", "expense_ratio": 0.15, "return_1y":  8.5, "risk_level": "중간", "dividend_yield": 1.8, "volatility": 15.2},
    {"name": "KODEX 미국S&P500TR",    "category": "해외주식", "expense_ratio": 0.05, "return_1y": 25.3, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 18.5},
    {"name": "ACE 미국배당다우존스",   "category": "배당",     "expense_ratio": 0.01, "return_1y": 12.1, "risk_level": "낮음", "dividend_yield": 3.5, "volatility": 10.3},
    {"name": "TIGER 2차전지테마",     "category": "테마",     "expense_ratio": 0.45, "return_1y":-15.2, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 35.7},
    {"name": "TIGER 국고채10년",      "category": "채권",     "expense_ratio": 0.07, "return_1y":  4.2, "risk_level": "낮음", "dividend_yield": 2.8, "volatility":  5.1},
    {"name": "KODEX 골드선물(H)",     "category": "원자재",   "expense_ratio": 0.68, "return_1y": 18.7, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 20.4},
    {"name": "KODEX 미국나스닥100TR", "category": "해외주식", "expense_ratio": 0.05, "return_1y": 32.1, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 25.3},
    {"name": "TIGER 미국반도체",      "category": "섹터",     "expense_ratio": 0.49, "return_1y": 45.0, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 38.2},
    {"name": "KODEX 단기채권PLUS",    "category": "채권",     "expense_ratio": 0.03, "return_1y":  3.8, "risk_level": "낮음", "dividend_yield": 3.2, "volatility":  1.5},
    {"name": "TIGER 리츠부동산",      "category": "리츠",     "expense_ratio": 0.29, "return_1y":  6.5, "risk_level": "중간", "dividend_yield": 4.8, "volatility": 12.8},
    {"name": "KODEX 200TR",           "category": "국내주식", "expense_ratio": 0.12, "return_1y":  9.1, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 14.9},
    {"name": "TIGER 고배당저변동",    "category": "배당",     "expense_ratio": 0.30, "return_1y":  7.2, "risk_level": "낮음", "dividend_yield": 5.2, "volatility":  8.7},
]

In [80]:
etf_df = pd.DataFrame(etf_data)
etf_df.head()

,name,category,expense_ratio,return_1y,risk_level,dividend_yield,volatility
0,KODEX 200,국내주식,0.15,8.5,중간,1.8,15.2
1,KODEX 미국S&P500TR,해외주식,0.05,25.3,중간,0.0,18.5
2,ACE 미국배당다우존스,배당,0.01,12.1,낮음,3.5,10.3
3,TIGER 2차전지테마,테마,0.45,-15.2,높음,0.0,35.7
4,TIGER 국고채10년,채권,0.07,4.2,낮음,2.8,5.1


In [81]:
risk_map = {"낮음" : 1, "중간" : 2, "높음" : 3}

def etf_to_vector(etf):
    return np.array([
        risk_map[etf['risk_level']] / 3,
        (etf['return_1y'] +30) /80,
        etf['dividend_yield'] / 6,
        etf['expense_ratio'],
        etf['volatility'] / 40
    ])

In [83]:
from numpy.linalg import norm
def cosine_sim(a, b):
    return float(np.dot(a,b) / (norm(a) * norm(b) + 1e-8))

In [84]:
item_vectors = np.array([etf_to_vector(e) for e in etf_data])
item_names = [e['name'] for e in etf_data]

In [85]:
item_vectors.shape

(12, 5)

In [86]:
def build_item_similarity(vectors):
    n = len(vectors)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = cosine_sim(vectors[i], vectors[j])
    return sim

In [87]:
item_sim = build_item_similarity(item_vectors)

In [89]:
# item_sim
def cbf_similar_items(target_name, top_k=5):
    idx = item_names.index(target_name)
    scores = [(item_names[j], item_sim[idx, j]) for j in range(len(item_names)) if j!=idx]
    return sorted(scores, key=lambda x:x[1], reverse=True)[:top_k]

In [90]:
for name, score in cbf_similar_items('KODEX 200', top_k=5):
    print(f" {score} | {name}")

 0.9499766521932036 | KODEX 200TR
 0.9417740369716172 | KODEX 미국나스닥100TR
 0.9317615934699991 | KODEX 미국S&P500TR
 0.9242601096073657 | TIGER 미국반도체
 0.9101145298708155 | TIGER 리츠부동산


In [91]:
a = [1,2,3,5,4]
a.sort()
a

[1, 2, 3, 4, 5]

In [92]:
a = [1,2,3,5,4]
sorted(a)
a

[1, 2, 3, 5, 4]

In [95]:
etf_data

[{'name': 'KODEX 200',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'return_1y': 8.5,
  'risk_level': '중간',
  'dividend_yield': 1.8,
  'volatility': 15.2},
 {'name': 'KODEX 미국S&P500TR',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'return_1y': 25.3,
  'risk_level': '중간',
  'dividend_yield': 0.0,
  'volatility': 18.5},
 {'name': 'ACE 미국배당다우존스',
  'category': '배당',
  'expense_ratio': 0.01,
  'return_1y': 12.1,
  'risk_level': '낮음',
  'dividend_yield': 3.5,
  'volatility': 10.3},
 {'name': 'TIGER 2차전지테마',
  'category': '테마',
  'expense_ratio': 0.45,
  'return_1y': -15.2,
  'risk_level': '높음',
  'dividend_yield': 0.0,
  'volatility': 35.7},
 {'name': 'TIGER 국고채10년',
  'category': '채권',
  'expense_ratio': 0.07,
  'return_1y': 4.2,
  'risk_level': '낮음',
  'dividend_yield': 2.8,
  'volatility': 5.1},
 {'name': 'KODEX 골드선물(H)',
  'category': '원자재',
  'expense_ratio': 0.68,
  'return_1y': 18.7,
  'risk_level': '중간',
  'dividend_yield': 0.0,
  'volatility': 20.4},
 {'name': 'KODEX 미국나스닥1

In [97]:
j

NameError: name 'j' is not defined

In [100]:
def cbf_diverse(target_name, top_k=5):
    idx = item_names.index(target_name)
    scores = [(j, item_sim[idx, j]) for j in range(len(item_names)) if j!=idx]
    scores.sort(key = lambda x : x[1], reverse=True)
    
    result, seen_cats = [], set()
    for j, score in scores:
        print(j)
        cat = etf_data[j]['category']
        if cat in seen_cats:
            continue
        result.append((item_names[j], round(score, 3), cat))
        seen_cats.add(cat)
        if len(result) >= top_k:
            break
    return result

In [101]:
cbf_diverse('KODEX 200', top_k=5)

10
6
1
7
9
4


[('KODEX 200TR', np.float64(0.95), '국내주식'),
 ('KODEX 미국나스닥100TR', np.float64(0.942), '해외주식'),
 ('TIGER 미국반도체', np.float64(0.924), '섹터'),
 ('TIGER 리츠부동산', np.float64(0.91), '리츠'),
 ('TIGER 국고채10년', np.float64(0.889), '채권')]